
===== UNIT III =====

LAB EXPERIMENTS




In [1]:
!pip install -q openai pandas

from google.colab import userdata
from openai import OpenAI

MAX_CALLS = 40
call_count = 0

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("GROQ_API_KEYS"),
)
MODEL = "llama-3.3-70b-versatile"


def ask_llm(prompt, system=None, temperature=0.3):
    """Send a prompt to Groq. Raises if daily call cap exceeded."""
    global call_count
    call_count += 1
    if call_count > MAX_CALLS:
        raise RuntimeError(f"Lab limit reached ({MAX_CALLS} calls). Stop and save your work.")
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

print("Setup complete. Model:", MODEL)


Setup complete. Model: llama-3.3-70b-versatile


In [2]:
# 1. Structured Format Prompting: Instruct the model to output information as bullet lists and Markdown tables (e.g., "List three benefits of daily exercise in a Markdown table with columns 'Benefit' and 'Description."); verify the output matches the requested structure.


article="""Retrieval-Augmented Generation (RAG) connects a language model to external documents. Documents are split into chunks and embedded. At query time, relevant chunks are retrieved and added to the prompt so the model answers from your data instead of guessing."""
cycle1=ask_llm("summarise this article:\n{article}")
print("===cycle1(open)===\n",cycle1)
cycle2_prompt="""summarize the article below in at most  80 words as exactly 4 bullet points.use section:problem| method|result|limitation|"""

{article}
cycle2=ask_llm(cycle2_prompt)
print(cycle2)
table_prompt="""list the  three benfits about daily exercise in a Markdown table with columns:|benfits|description|"""
table_output=ask_llm(table_prompt)
print("===table===\n",table_prompt)
from IPython.display import Markdown,display
display(Markdown(table_output))


===cycle1(open)===
 There is no article provided. Please paste the article you'd like me to summarize, and I'll be happy to assist you.
Since there's no article provided, I'll create a sample summary. 
* Problem: Unknown
* Method: Unspecified
* Result: Undisclosed
* Limitation: Unaddressed
===table===
 list the  three benfits about daily exercise in a Markdown table with columns:|benfits|description|


| Benfits | Description |
| --- | --- |
| Weight Management | Regular daily exercise helps burn calories and maintain a healthy weight, reducing the risk of obesity and related diseases. |
| Improved Mental Health | Daily exercise releases endorphins, also known as "feel-good" hormones, which can boost mood, reduce stress, and alleviate symptoms of anxiety and depression. |
| Increased Energy | Engaging in daily physical activity can increase energy levels, enhance sleep quality, and improve overall physical function, allowing individuals to tackle daily tasks with more vigor and efficiency. |

In [3]:
 # 2. JSON/YAML Generation: Provide a brief dataset description (e.g., three books with title, author, publication year) and prompt the model to produce valid JSON or YAML; use a parser to validate syntax and refine the prompt if errors occur.

import json
import yaml

dataset = """
Books:
1. The Alchemist - Paulo Coelho - 1988
2. Atomic Habits - James Clear - 2018
3. Clean Code - Robert C. Martin - 2008
"""

# JSON
json_prompt = f"""
Convert the following dataset into valid JSON.
Return ONLY JSON.

{dataset}
"""

json_output = ask_llm(json_prompt)
print("JSON Output:\n", json_output)

try:
    parsed_json = json.loads(json_output)
    print("✅ Valid JSON")
except json.JSONDecodeError:
    print("❌ Invalid JSON")
    print("Refining prompt...")
    json_output = ask_llm(json_prompt + "\nDo not use markdown or explanations. Return only valid JSON.")
    print(json_output)

# YAML
yaml_prompt = f"""
Convert the following dataset into valid YAML.
Return ONLY YAML.

{dataset}
"""

yaml_output = ask_llm(yaml_prompt)
print("\nYAML Output:\n", yaml_output)

try:
    parsed_yaml = yaml.safe_load(yaml_output)
    print("✅ Valid YAML")
except yaml.YAMLError:
    print("❌ Invalid YAML")
    print("Refining prompt...")
    yaml_output = ask_llm(yaml_prompt + "\nDo not use markdown or explanations. Return only valid YAML.")
    print(yaml_output)


JSON Output:
 ```
{
  "Books": [
    {
      "title": "The Alchemist",
      "author": "Paulo Coelho",
      "year": 1988
    },
    {
      "title": "Atomic Habits",
      "author": "James Clear",
      "year": 2018
    },
    {
      "title": "Clean Code",
      "author": "Robert C. Martin",
      "year": 2008
    }
  ]
}
```
❌ Invalid JSON
Refining prompt...
[
  {
    "title": "The Alchemist",
    "author": "Paulo Coelho",
    "year": 1988
  },
  {
    "title": "Atomic Habits",
    "author": "James Clear",
    "year": 2018
  },
  {
    "title": "Clean Code",
    "author": "Robert C. Martin",
    "year": 2008
  }
]

YAML Output:
 ---
Books:
  - title: The Alchemist
    author: Paulo Coelho
    year: 1988
  - title: Atomic Habits
    author: James Clear
    year: 2018
  - title: Clean Code
    author: Robert C. Martin
    year: 2008
✅ Valid YAML


In [4]:
#  3 .Chain-of-Thought & Task Decomposition: Present a multi-step problem (e.g., a logic puzzle) and apply zero-shot CoT prompting (e.g., "Let's think step by step. Explain your reasoning before the final answer."); separately, decompose the problem into sequential sub-questions, collect partial answers, combine them, and compare accuracy against a direct-answer baseline.

# Multi-step Problem
problem = """
A train travels 120 km in 2 hours and then 180 km in 3 hours.
What is the average speed of the train for the entire journey?
"""

# ---------------- Baseline ----------------
baseline_prompt = f"""
Answer the following question directly.

{problem}
"""

print("===== BASELINE =====")
baseline = ask_llm(baseline_prompt)
print(baseline)

# ---------------- Zero-Shot Chain of Thought ----------------
cot_prompt = f"""
Solve the following problem.

Let's think step by step.
Explain your reasoning before giving the final answer.

{problem}
"""

print("\n===== ZERO-SHOT CHAIN OF THOUGHT =====")
cot = ask_llm(cot_prompt)
print(cot)

# ---------------- Task Decomposition ----------------

print("\n===== TASK DECOMPOSITION =====")

q1 = ask_llm("A train travels 120 km in 2 hours. What is the distance?")
print("Step 1:", q1)

q2 = ask_llm("A train travels 180 km in 3 hours. What is the distance?")
print("Step 2:", q2)

q3 = ask_llm("The train traveled 120 km in 2 hours and 180 km in 3 hours. What is the total distance?")
print("Step 3:", q3)

q4 = ask_llm("The train traveled for 2 hours and then 3 hours. What is the total time?")
print("Step 4:", q4)

final_prompt = """
The train traveled a total distance of 300 km in a total time of 5 hours.
Calculate the average speed.

Return only the final answer.
"""

final_answer = ask_llm(final_prompt)
print("Final Answer:", final_answer)

print("\n===== COMPARISON =====")
print("1. Baseline: Gives a direct answer with no reasoning.")
print("2. Zero-Shot CoT: Explains the reasoning step by step before the final answer.")
print("3. Task Decomposition: Solves the problem using smaller sub-questions and combines the results, improving clarity and accuracy.")

===== BASELINE =====
To find the average speed of the train for the entire journey, we need to calculate the total distance traveled and the total time taken.

Total distance = 120 km + 180 km = 300 km
Total time = 2 hours + 3 hours = 5 hours

Average speed = Total distance / Total time
= 300 km / 5 hours
= 60 km/h

So, the average speed of the train for the entire journey is 60 km/h.

===== ZERO-SHOT CHAIN OF THOUGHT =====
To find the average speed of the train for the entire journey, we need to follow these steps:

1. Calculate the total distance traveled by the train.
2. Calculate the total time taken by the train to travel the total distance.
3. Use the formula for average speed: Average Speed = Total Distance / Total Time.

Let's break it down:

- The train travels 120 km in 2 hours.
- Then, it travels 180 km in 3 hours.

Step 1: Calculate the total distance traveled by the train.
Total Distance = 120 km + 180 km = 300 km.

Step 2: Calculate the total time taken by the train to tr